# 単語分割を文字単位の系列ラベリングとして解く

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_sequence_labeling/char_segmentation.ipynb)

NLP クックブック 13-07 ／ 系列ラベリング

日本語の単語分割を「各文字が単語の先頭 (B) か途中 (I) か」を当てる問題として、CRF で学習したい。

In [1]:
!pip install -q sklearn-crfsuite fugashi unidic-lite regex

In [2]:
import io, re, urllib.request, zipfile
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

# MeCab の分割結果を正解とみなす (本来は人手で分割したコーパスを使う)
tagger = Tagger()
text = aozora("https://www.aozora.gr.jp/cards/000148/files/752_ruby_2438.zip")
sents = [[w.surface for w in tagger(s)] for s in re.split(r"(?<=。)|\n", text) if s and s.strip()]
train, test = sents[:2200], sents[2200:]

def to_chars_tags(words):
    chars = [c for w in words for c in w]
    tags = [t for w in words for t in ["B"] + ["I"] * (len(w) - 1)]
    return chars, tags

In [3]:
import regex
import sklearn_crfsuite

def char_type(c):
    for name in ["Han", "Hiragana", "Katakana", "Latin", "Nd"]:
        if regex.match(rf"\p{{{name}}}", c):
            return name
    return "Other"

def features(chars, i):
    f = {"bias": 1.0}
    for d in range(-2, 3):          # 前後 2 文字の文字と文字種
        if 0 <= i + d < len(chars):
            f[f"c{d}"] = chars[i + d]
            f[f"t{d}"] = char_type(chars[i + d])
    for d in (-1, 0):               # 文字 bigram と文字種の変わり目
        if 0 <= i + d and i + d + 1 < len(chars):
            f[f"bi{d}"] = chars[i + d] + chars[i + d + 1]
            f[f"tt{d}"] = char_type(chars[i + d]) + ">" + char_type(chars[i + d + 1])
    return f

def dataset(sents):
    X, y = [], []
    for words in sents:
        chars, tags = to_chars_tags(words)
        X.append([features(chars, i) for i in range(len(chars))]); y.append(tags)
    return X, y

X_train, y_train = dataset(train)
X_test, y_test = dataset(test)
crf = sklearn_crfsuite.CRF(c1=0.1, c2=0.1, max_iterations=60).fit(X_train, y_train)

In [4]:
def segment(sentence):
    chars = list(sentence)
    tags = crf.predict_single([features(chars, i) for i in range(len(chars))])
    return "".join((" " if t == "B" and i else "") + c for i, (c, t) in enumerate(zip(chars, tags))).split()

def word_f1(gold_sents, pred_sents):
    """単語単位の F1: 開始位置と終了位置が両方一致した単語だけを正解とする"""
    tp = n_gold = n_pred = 0
    for gold, pred in zip(gold_sents, pred_sents):
        spans = lambda ws: {(sum(map(len, ws[:i])), sum(map(len, ws[:i + 1]))) for i in range(len(ws))}
        g, p = spans(gold), spans(pred)
        tp += len(g & p); n_gold += len(g); n_pred += len(p)
    prec, rec = tp / n_pred, tp / n_gold
    return 2 * prec * rec / (prec + rec)

pred = [segment("".join(ws)) for ws in test]
print("単語 F1:", round(word_f1(test, pred), 4))
print(segment("親譲りの無鉄砲で小供の時から損ばかりしている。"))
print(segment("最新のスマートフォンで自然言語処理を勉強する。"))

単語 F1: 0.8616
['親譲り', 'の', '無鉄砲', 'で', '小供', 'の', '時', 'から', '損', 'ばかり', 'し', 'て', 'いる', '。']
['最新', 'の', 'スマートフォン', 'で', '自然', '言語', '処理', 'を', '勉強', 'する', '。']


- 辞書を使わない「点予測」に近い方式 (KyTea など) の簡易版。学習データにない分野の語 (2 文目) では誤りが増える。
- 中国語の単語分割では、B/M/E/S (先頭・中間・末尾・単独) の 4 タグが標準的。